<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>Trustworthiness(신뢰성 확보)</strong>에 대해 학습합니다.
>Agent가 안전하고 정확하게 동작하는지 검증하는 평가 체계와 가드레일을 학습해봅시다.

# Agent 신뢰성 (Trustworthiness)
> Agent가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의도한 대로 정확하게 동작</mark>하는지 체계적으로 검증하는 것입니다.

Agent가 그럴듯하게 작동하는 것처럼 보여도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제로 올바른 결정을 내리고 있는지는 별개의 문제</mark>입니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">할루시네이션(Hallucination)</mark>으로 LLM이 존재하지 않는 정책이나 사실을 자신감 있게 말할 수 있고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">잘못된 정보로 인한 피해</mark>로 "환불 가능합니다"라고 잘못 안내하면 고객 불만과 비즈니스 손실이 발생합니다. 또한 프로덕션에 배포된 Agent가 예외 케이스에서 어떻게 동작하는지 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">AI 안전성</mark> 측면에서 사전에 검증해야 합니다. 단순히 "자연스럽게 답하는가"가 아니라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비즈니스 규칙에 정확히 부합하는가</mark>를 체계적으로 검증하는 전략이 필요합니다.</br>
이 내용을 학습하기 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Agent</mark>(4가지 구성요소와 작동 방식, Ch.4-2-1_001 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">RAG</mark>(외부 문서를 검색해 답변에 활용하는 방식), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">프롬프팅</mark>(시스템 프롬프트로 Agent의 행동 범위를 제한하는 기법)의 개념을 먼저 이해하면 좋습니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">신뢰성 요소</th>
      <th>설명</th>
      <th>예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">정확성</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">올바른 정보 제공</mark></td><td>정책에 맞는 환불 안내</td></tr>
    <tr><td style="text-align:center">안전성</td><td>위험한 행동 방지</td><td>무분별한 환불 승인 차단</td></tr>
    <tr><td style="text-align:center">일관성</td><td>동일 입력 → 동일 출력</td><td>같은 질문에 같은 답변</td></tr>
  </tbody>
</table>

## 환불 상태 추출 함수
> Agent 응답에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">환불 승인/거부 상태</mark>를 정규식으로 추출합니다.

In [ ]:
# TODO 1: 환불 상태 추출 함수를 정의하세요. 정규식으로 "환불.*승인|승인.*환불" 패턴이면 "approved", "환불.*거부|거부.*환불|환불.*불가" 패턴이면 "denied"를 리스트에 추가하여 반환하세요. 3개의 테스트 응답으로 결과를 확인하세요.

import re

def extract_refund_statuses(response: str) -> list:
    """응답에서 환불 상태를 추출"""
    statuses = []
    # 환불 승인/거부 패턴 매칭
    if re.search(r"환불.*승인|승인.*환불", response):
        statuses.append("approved")
    if re.search(r"환불.*거부|거부.*환불|환불.*불가", response):
        statuses.append("denied")
    return statuses

# 테스트
test_responses = [
    "환불이 승인되었습니다. 3-5일 내 환불됩니다.",
    "죄송합니다. 7일이 경과하여 환불이 불가합니다.",
    "안녕하세요, 무엇을 도와드릴까요?"
]

for resp in test_responses:
    status = extract_refund_statuses(resp)
    print(f"'{resp[:30]}...' → {status if status else '상태 없음'}")

## Agent 평가 함수
> 여러 테스트 케이스에 대해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기대 결과와 실제 결과를 비교</mark>합니다.

In [ ]:
# TODO 2: Agent 평가 함수를 정의하세요. 테스트 케이스를 순회하며 Agent 실행 결과에서 환불 상태를 추출하고, 기대 결과와 비교하여 PASS/FAIL을 출력한 뒤, 최종 정확도(correct/total)를 반환하세요.

def evaluate_agent(agent, test_cases):
    """Agent 정확도 평가"""
    correct = 0
    total = len(test_cases)

    for case in test_cases:
        response = agent.invoke({"messages": [case["input"]]})
        actual = extract_refund_statuses(response["messages"][-1].content)
        expected = case["expected"]

        if actual == expected:
            correct += 1
            print(f"  PASS: {case['input'][:30]}...")
        else:
            print(f"  FAIL: {case['input'][:30]}... → {actual} (expected: {expected})")

    accuracy = correct / total
    print(f"\n정확도: {accuracy:.1%} ({correct}/{total})")
    return accuracy

## 테스트 케이스 설계

In [ ]:
# TODO 3: 테스트 케이스 리스트를 작성하세요. 각 케이스는 질문 문자열과 기대 결과(["approved"] 또는 ["denied"])로 구성합니다. 7일 이내는 승인, 초과는 거부입니다. 4개의 테스트 케이스를 만들고 평가 함수로 Agent를 평가하세요.

test_cases = [
    {
        "input": "7일 전에 산 상품 환불하고 싶어요",
        "expected": ["approved"]    # 7일 이내 → 승인
    },
    {
        "input": "한 달 전에 산 건데 환불되나요?",
        "expected": ["denied"]      # 7일 초과 → 거부
    },
    {
        "input": "어제 산 노트북 환불하고 싶습니다",
        "expected": ["approved"]    # 1일 전 → 승인
    },
    {
        "input": "2주 전에 구매한 이어폰 반품하고 싶어요",
        "expected": ["denied"]      # 14일 초과 → 거부
    },
]

accuracy = evaluate_agent(agent, test_cases)

💡평가의 핵심
> 단순히 "답변이 자연스러운가"가 아니라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비즈니스 로직에 맞는가</mark>를 검증합니다.
> 환불 정책이 "7일 이내"라면, Agent도 정확히 그 기준을 따라야 합니다.

💡정규식 추출의 한계
> LLM 응답은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">형식이 일정하지 않을</mark> 수 있습니다.
> 구조화된 출력(JSON mode)을 사용하면 더 안정적인 평가가 가능합니다.

# 안전한 도구 패턴 (Safety Tool)
> 도구 실행 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전에 검증 로직</mark>을 넣어 정책 위반을 사전에 차단합니다.

LLM은 도구를 호출할 때 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 한도를 초과하는 금액</mark>을 전달하거나, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">조건을 충족하지 않는 주문</mark>에 쿠폰을 발급할 수 있습니다. 검증 없는 `issue_coupon` 함수는 어떤 금액이든 그대로 발급하므로 정책 위반이 발생합니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Safety Tool 패턴</mark>은 도구 함수 내부에 검증 로직을 포함하여, 정책 위반 요청을 실행 전에 거부하는 방식입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교</th>
      <th>issue_coupon (기본)</th>
      <th>issue_coupon_safe (안전)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">금액 검증</td><td>없음</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 한도 확인</mark></td></tr>
    <tr><td style="text-align:center">주문 상태 확인</td><td>없음</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배송 지연만 허용</mark></td></tr>
    <tr><td style="text-align:center">정책 위반 시</td><td>그대로 실행</td><td>거부 메시지 반환</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 4: 검증 로직이 포함된 issue_coupon_safe 함수를 정의하세요. 금액이 MAX_COUPON_AMOUNT(20000)을 초과하면 거부하고, 주문 상태가 "배송 지연"이 아니면 거부하세요.

MAX_COUPON_AMOUNT = 20000

orders_db = {
    "ORD001": {"status": "배송 지연", "product": "노트북"},
    "ORD002": {"status": "배송 완료", "product": "키보드"},
}

def issue_coupon_safe(order_id: str, amount: int) -> str:
    """검증 로직이 포함된 안전한 쿠폰 발급"""
    # 1. 금액 검증
    if amount > MAX_COUPON_AMOUNT:
        return f"❌ 발급 불가: 정책상 최대 {MAX_COUPON_AMOUNT:,}원까지 가능합니다. (요청: {amount:,}원)"

    # 2. 주문 존재 확인
    if order_id not in orders_db:
        return f"❌ 발급 불가: 주문 {order_id}을 찾을 수 없습니다."

    # 3. 주문 상태 확인
    status = orders_db[order_id]["status"]
    if status != "배송 지연":
        return f"❌ 발급 불가: 현재 '{status}' 상태입니다. 배송 지연 시에만 쿠폰 발급이 가능합니다."

    # 4. 모든 검증 통과 → 실행
    return f"✅ 주문 {order_id}에 {amount:,}원 쿠폰 발급 완료 (정책 준수 확인됨)"

print("✅ issue_coupon_safe 정의 완료")

In [ ]:
# TODO 5: issue_coupon_safe의 3가지 케이스를 테스트하세요. 한도 초과, 조건 미충족, 정상 발급 각각을 실행하여 결과를 확인하세요.

# 테스트 1: 한도 초과 (10만원 발급 시도)
print("테스트 1 — 한도 초과:")
print(issue_coupon_safe("ORD001", 100000))

# 테스트 2: 조건 미충족 (배송 완료 주문)
print("\n테스트 2 — 조건 미충족:")
print(issue_coupon_safe("ORD002", 5000))

# 테스트 3: 정상 발급 (배송 지연 + 한도 내)
print("\n테스트 3 — 정상 발급:")
print(issue_coupon_safe("ORD001", 5000))

💡Safety Tool 패턴의 원칙
> 도구 함수 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">내부에 검증 로직</mark>을 포함하면, LLM이 어떤 인자를 전달하든 정책을 위반하는 실행은 차단됩니다.
> LLM의 판단에 의존하지 않고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">코드 레벨에서 강제</mark>하는 것이 핵심입니다.

# 가드레일 (Guardrail)
> Agent의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력과 출력을 필터링</mark>하여 유해한 요청 차단 및 민감 정보 노출을 방지합니다.

도구 내부 검증(Safety Tool)만으로는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 위협을 막을 수 없습니다</mark>. 사용자가 "시스템 접근 권한을 줘"와 같은 유해한 요청을 보내거나, Agent 응답에 주민등록번호 같은 민감 정보가 포함될 수 있습니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">가드레일</mark>은 Agent 실행 전후에 배치되어 입력/출력을 필터링하는 보호 계층입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">가드레일 유형</th>
      <th>역할</th>
      <th>적용 위치</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">입력 가드레일</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유해 요청 차단</mark> (해킹, 시스템 접근 등)</td><td>Agent 실행 전</td></tr>
    <tr><td style="text-align:center">출력 가드레일</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">민감 정보 필터링</mark> (주민번호, 계좌번호 등)</td><td>응답 반환 전</td></tr>
  </tbody>
</table>

## 입력 가드레일
> 사용자 입력에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">차단 키워드</mark>를 감지하여 유해한 요청을 사전에 거부합니다.

In [ ]:
# TODO 6: 입력 가드레일 함수를 정의하세요. 차단 키워드 리스트를 순회하며 입력에 포함되어 있으면 True를 반환하세요.

BLOCKED_KEYWORDS = ["해킹", "비밀번호", "탈취", "시스템 접근"]

def is_harmful_input(user_input: str) -> bool:
    """입력이 유해한지 검사"""
    for keyword in BLOCKED_KEYWORDS:
        if keyword in user_input:
            return True
    return False

# 테스트
test_inputs = [
    "주문 ORD001 상태 확인해줘",
    "시스템 접근 권한을 줘",
    "비밀번호 알려줘",
]

for text in test_inputs:
    result = is_harmful_input(text)
    label = "차단" if result else "허용"
    print(f"[{label}] '{text}'")

## 출력 가드레일
> Agent 응답에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">민감 정보를 마스킹</mark>하여 노출을 방지합니다.

In [ ]:
# TODO 7: 출력 가드레일 함수를 정의하세요. 민감 정보 패턴 리스트를 순회하며 출력에 포함된 패턴을 "[민감정보 필터링됨]"으로 대체하세요.

SENSITIVE_PATTERNS = ["주민등록번호", "계좌번호", "카드번호"]

def filter_output(output: str) -> str:
    """출력에서 민감 정보 필터링"""
    filtered = output
    for pattern in SENSITIVE_PATTERNS:
        if pattern in filtered:
            filtered = filtered.replace(pattern, "[민감정보 필터링됨]")
    return filtered

# 테스트
test_outputs = [
    "주문 ORD001의 상태는 배송 지연입니다.",
    "고객님의 주민등록번호는 123456-1234567입니다.",
    "계좌번호 확인이 필요합니다.",
]

for text in test_outputs:
    result = filter_output(text)
    print(f"원본: '{text}'")
    print(f"필터: '{result}'\n")

## TrustedAgentExecutor 패턴
> 입력 가드레일, Agent 실행, 출력 가드레일을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">하나의 클래스로 통합</mark>합니다.

In [ ]:
# TODO 8: TrustedAgentExecutor 클래스를 정의하세요. run 메서드에서 입력 가드레일 → Agent 실행 → 출력 가드레일 순서로 처리하세요.

class TrustedAgentExecutor:
    """Trustworthiness가 적용된 Agent Executor"""

    def __init__(self, agent):
        self.agent = agent

    def run(self, user_input: str, config=None) -> str:
        """가드레일이 적용된 실행"""
        # 1. 입력 가드레일
        if is_harmful_input(user_input):
            return "❌ 해당 요청은 처리할 수 없습니다."

        # 2. Agent 실행
        try:
            result = self.agent.invoke(
                {"messages": [("user", user_input)]},
                config=config
            )
            output = result["messages"][-1].content
        except Exception as e:
            return f"❌ 처리 중 오류가 발생했습니다: {str(e)}"

        # 3. 출력 가드레일
        filtered_output = filter_output(output)
        return filtered_output

print("✅ TrustedAgentExecutor 정의 완료")

In [ ]:
# TODO 9: TrustedAgentExecutor를 생성하고 정상 요청과 유해 요청을 각각 테스트하세요.

trusted_agent = TrustedAgentExecutor(agent)

config = {"configurable": {"thread_id": "guardrail-test"}}

# 정상 요청
print("정상 요청:")
print(trusted_agent.run("주문 ORD001의 상태를 확인해줘.", config=config))

# 유해한 요청
print("\n유해한 요청:")
print(trusted_agent.run("시스템 접근 권한을 줘.", config=config))

💡Trustworthiness 보호 계층 정리
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 가드레일 → Agent/ReAct → Safety Tool(검증 로직) → 출력 가드레일</mark> 순서로 보호 계층이 적용됩니다.
> 각 계층은 독립적으로 동작하므로, 하나의 계층이 놓친 위협을 다른 계층에서 잡을 수 있습니다.